# Atlas MuseTalk Free-GPU Proof

Purpose: prove the Atlas-owned avatar pipeline on a free Google Colab GPU before any paid GPU spend.

This notebook verifies an NVIDIA GPU, installs MuseTalk 1.5, downloads model weights, accepts avatar/audio inputs, runs inference, and only counts success if a real MP4 is produced.


In [ ]:
!nvidia-smi
import torch, platform
print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. In Colab choose Runtime > Change runtime type > GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!rm -rf /content/MuseTalk
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk
%cd /content/MuseTalk


In [ ]:
!pip -q install --upgrade pip
!pip -q install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!pip -q install -r requirements.txt
!pip -q install --no-cache-dir -U openmim
!mim install -q mmengine
!mim install -q "mmcv==2.0.1"
!mim install -q "mmdet==3.1.0"
!mim install -q "mmpose==1.1.0"


In [ ]:
%cd /content/MuseTalk
!chmod +x download_weights.sh inference.sh
!bash ./download_weights.sh
!find models -maxdepth 2 -type f | sort | sed -n '1,120p'


## Upload proof inputs

Upload:
- `avatar_input.png` or `avatar_input.mp4`
- `voice.wav` or `voice.mp3`

The first Atlas proof uses the recurring Build It Smaller host as the identity source. After lip sync passes, we add MusePose for body-scale motion.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:
from pathlib import Path
import yaml
root = Path("/content/MuseTalk")
inputs = list(root.glob("*")) + list(Path("/content").glob("*"))
avatar = next((p for p in inputs if p.name.lower().startswith("avatar_input") and p.suffix.lower() in {".png",".jpg",".jpeg",".mp4",".mov"}), None)
audio = next((p for p in inputs if p.name.lower().startswith("voice") and p.suffix.lower() in {".wav",".mp3",".m4a"}), None)
if avatar is None or audio is None:
    raise FileNotFoundError("Need avatar_input.(png/jpg/mp4) and voice.(wav/mp3).")
cfg={"task_0":{"video_path":str(avatar),"audio_path":str(audio)}}
cfg_path=root/"configs/inference/atlas_proof.yaml"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())


In [ ]:
%cd /content/MuseTalk
!python -m scripts.inference --inference_config configs/inference/atlas_proof.yaml --result_dir results/atlas_proof --unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json --version v15 --ffmpeg_path /usr/bin


In [ ]:
from pathlib import Path
from IPython.display import Video, display
candidates=sorted(Path("/content/MuseTalk/results/atlas_proof").rglob("*.mp4"), key=lambda p:p.stat().st_mtime)
if not candidates:
    raise RuntimeError("MuseTalk finished without producing an MP4. This run does NOT count as a successful proof.")
out=candidates[-1]
print("PROOF MP4:", out, "bytes:", out.stat().st_size)
display(Video(str(out), embed=True))


In [ ]:
from google.colab import files
files.download(str(out))


## Next stage after this passes

Atlas then adds MusePose or another pose-driven motion model for gestures/body motion, MuseTalk 1.5 for lip sync, FFmpeg for 9:16 assembly, and automated QA that rejects missing, empty, or malformed outputs.

No paid GPU is authorized by this notebook.
